# URL 斷詞 · 解碼 · 多語言翻譯 · 貼標 (tagging)

把一條 URL 變成 **乾淨、有標籤、看得懂** 的 token，當作惡意 URL 偵測的特徵。

### 核心觀念
**URL 不是自然語言句子** → 不要用「中文斷詞 / BERT 句子分詞」硬套整條 URL。
正確的工具鏈是四件事拼起來：

> 智慧解碼 → 結構解析(含後綴) → 異常字段過濾 → 斷字/翻譯 → 上標籤

### 整條 pipeline
```
原始 URL
  │
① 智慧解碼   %E5%84%BF → 兒  (UTF-8/GBK/Big5/Punycode)
② 結構解析   scheme / 子網域 / 網域 / 後綴(TLD) / 路徑 / 副檔名
③ 異常過濾   丟掉 hash / UUID / base64 / 高熵亂碼   ← 先做，避免亂碼被硬斷
④ 斷字       英文 wordsegment、中文 jieba
⑤ 多語言翻譯  非英文字段 → 英文 → 再斷詞 → 標 label_en
⑥ 上標籤     每個 token 標角色：tld / domain / cjk / word / label_en / anomaly ...
```
所有邏輯都在 `url_features.py`，這本 notebook 一段段示範。

## 0. 載入資料與模組

In [1]:
# 自動重載：之後修改 url_features.py 會自動生效，免得 kernel 用到舊版
%load_ext autoreload
%autoreload 2

import warnings, os
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import pandas as pd
import url_features as uf   # 我們的特徵工程模組

df = pd.read_csv("./data/malicious_phish.csv")
print("資料筆數:", len(df))
df["type"].value_counts()

資料筆數: 651191


type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

In [2]:
# 各類別看幾個樣本，感受 URL 的多樣性
for t in df["type"].unique():
    print("===", t, "===")
    for u in df[df["type"] == t]["url"].head(3):
        print("  ", u)
    print()

=== phishing ===
   br-icloud.com.br
   signin.eby.de.zukruygxctzmmqi.civpro.co.za
   http://www.marketingbyinternet.com/mo/e56508df639f6ce7d55c81ee3fcd5ba8/

=== benign ===
   mp3raid.com/music/krizz_kaliko.html
   bopsecrets.org/rexroth/cr/1.htm
   http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html

=== defacement ===
   http://www.garage-pirenne.be/index.php?option=com_content&view=article&id=70&vsig70_0=15
   http://adventure-nicaragua.net/index.php?option=com_mailto&tmpl=component&link=aHR0cDovL2FkdmVudHVyZS1uaWNhcmFndWEubmV0L2luZGV4LnBocD9vcHRpb249Y29tX2NvbnRlbnQmdmlldz1hcnRpY2xlJmlkPTQ3OmFib3V0JmNhdGlkPTM2OmRlbW8tYXJ0aWNsZXMmSXRlbWlkPTU0
   http://www.pashminaonline.com/pure-pashminas

=== malware ===
   http://www.824555.com/app/member/SportOption.php?uid=guest&langx=gb
   http://9779.info/%E5%84%BF%E7%AB%A5%E7%AB%8B%E4%BD%93%E7%BA%B8%E8%B4%B4%E7%94%BB/
   http://9779.info/%E6%A0%91%E5%8F%B6%E7%B2%98%E8%B4%B4%E

## ① 智慧解碼 smart_decode

你的 malware 樣本常見 `%E5%84%BF...` 這種——其實是**中文(或其他語言)被編碼成 %數字符號**。
必須先解回原文，後面的斷詞/翻譯才有意義。`smart_decode` 會自動嘗試 UTF-8 → GBK → Big5 → 日韓編碼，
並處理 `xn--` Punycode 國際化域名。

In [3]:
samples = [
    "http://9779.info/%E5%84%BF%E7%AB%A5%E7%AB%8B%E4%BD%93%E7%BA%B8%E8%B4%B4%E7%94%BB/",
    "http://example.com/%E6%8A%98%E7%BA%B8%E6%89%87%E5%AD%90",
    "br-icloud.com.br",   # 沒有編碼 → 原樣回傳
]
for u in samples:
    decoded, enc = uf.smart_decode(u)
    print(f"enc={str(enc):6s} | {u}\n          → {decoded}\n")

enc=utf-8  | http://9779.info/%E5%84%BF%E7%AB%A5%E7%AB%8B%E4%BD%93%E7%BA%B8%E8%B4%B4%E7%94%BB/
          → http://9779.info/儿童立体纸贴画/

enc=utf-8  | http://example.com/%E6%8A%98%E7%BA%B8%E6%89%87%E5%AD%90
          → http://example.com/折纸扇子

enc=None   | br-icloud.com.br
          → br-icloud.com.br



## ② 結構解析 + 後綴(TLD) parse_structure

用 `urllib.parse` + `tldextract`(公共後綴清單 PSL) 把 URL 拆開。
**後綴一定要用 PSL**：`forums.bbc.co.uk` 的後綴是 `co.uk`(不是 `uk`)、`br-icloud.com.br` 是 `com.br`。
路徑副檔名(`.php`/`.html`)另外抓。

In [4]:
rows = [uf.parse_structure(u) for u in [
    "http://forums.bbc.co.uk/docs/guide.php?v=1",
    "br-icloud.com.br",
    "http://www.garage-pirenne.be/index.php?option=com_content",
    "http://37.49.226.178/x.sh4",   # IP 當主機
]]
pd.DataFrame(rows)[["host","subdomain","domain","suffix","registered_domain","file_ext","has_ip_host"]]

,host,subdomain,domain,suffix,registered_domain,file_ext,has_ip_host
0,forums.bbc.co.uk,forums,bbc,co.uk,bbc.co.uk,php,False
1,br-icloud.com.br,,br-icloud,com.br,br-icloud.com.br,NaN,False
2,www.garage-pirenne.be,www,garage-pirenne,be,garage-pirenne.be,php,False
3,37.49.226.178,,37.49.226.178,,,sh4,True


## ③ 異常字段過濾 is_anomalous（你的步驟3）

隨機 hash / UUID / base64 / 一長串數字，不該拿去斷詞（會被硬切成垃圾）。
**重點**：純靠熵會誤判——真實長黏字 `summaryauthpolicyagreement`(熵 3.84) 比 MD5(3.64) 還高。
所以我們加了**母音比例護欄**：像真實單字(母音比 0.26~0.62)的就不丟。

In [5]:
tests = [
    ("summaryauthpolicyagreement", "真實英文長黏字 → 留"),
    ("marketingbyinternet",        "黏字 → 留(交給斷字)"),
    ("c4d12146ebce8e1684d3542308399779", "MD5 → 丟"),
    ("550e8400-e29b-41d4-a716-446655440000", "UUID → 丟"),
    ("zukruygxctzmmqi",            "隨機子網域 → 丟"),
    ("1234567890123",             "長數字ID → 丟"),
]
print(f"{'token':38s}{'異常?':6s}{'母音比':7s}{'熵':6s}說明")
for t, why in tests:
    v = sum(c.lower() in 'aeiou' for c in t)/len(t) if t.isalpha() else 0
    print(f"{t:38s}{str(uf.is_anomalous(t)):6s}{v:<7.2f}{uf.shannon_entropy(t):<6.2f}{why}")

token                                 異常?   母音比    熵     說明
summaryauthpolicyagreement            False 0.38   3.84  真實英文長黏字 → 留
marketingbyinternet                   False 0.32   3.29  黏字 → 留(交給斷字)
c4d12146ebce8e1684d3542308399779      True  0.00   3.69  MD5 → 丟
550e8400-e29b-41d4-a716-446655440000  True  0.00   3.39  UUID → 丟
zukruygxctzmmqi                       True  0.20   3.51  隨機子網域 → 丟
1234567890123                         True  0.00   3.24  長數字ID → 丟


## ④ 斷字 segment_token — 英文 wordsegment、中文 jieba

In [6]:
for t in ["marketingbyinternet", "garagepirenne", "儿童立体纸贴画", "administrator"]:
    sub, method = uf.segment_token(t)
    print(f"{t:24s} →  {sub}   [{method}]")

marketingbyinternet      →  ['marketing', 'by', 'internet']   [wordsegment]
garagepirenne            →  ['garage', 'pi', 'renne']   [wordsegment]


儿童立体纸贴画                  →  ['儿童', '立体', '纸', '贴画']   [jieba]
administrator            →  ['administrator']   [wordsegment]


## ⑤ 多語言翻譯 translate_many — 任何語言 → 英文

用單一模型 `Helsinki-NLP/opus-mt-mul-en`，涵蓋數十種來源語言 → 英文（離線、有快取）。
第一次執行會載入模型(~300MB)，之後很快。

In [7]:
demo = [
    "儿童立体纸贴画",          # 簡體中文
    "兒童立體紙貼畫",          # 繁體中文
    "Lebensmittelüberwachung", # 德文
    "ファイルをダウンロード",     # 日文
    "administrador",           # 西班牙文
]
for src, en in zip(demo, uf.translate_many(demo)):
    print(f"{src:24s} → {en}")

儿童立体纸贴画                  → Children's stand-alone paper posters
兒童立體紙貼畫                  → Children's Picture
Lebensmittelüberwachung  → Food monitoring
ファイルをダウンロード              → Download File
administrador            → admin


## ⑥ 整合：tokenize_url 完整貼標

一條 URL 進去，吐出**每個 token 的角色標籤**。
`translate=True` 會把非英文字段翻成英文、再斷詞，標成 `label_en`（原文 token 仍保留）。
tag：`scheme/subdomain/tld/domain/file_ext/word/digit/cjk/label_en/anomaly`

In [8]:
def show(url):
    f = uf.tokenize_url(url, translate=True)
    print("URL :", url)
    print("解碼:", f["decoded_url"])
    if f["translations"]:
        print("翻譯:", f["translations"])
    print("後綴(TLD):", repr(f["suffix"]), "| 網域:", repr(f["domain"]), "| 副檔名:", repr(f["file_ext"]))
    print("帶標籤 token:")
    for tok, tag in f["tagged_tokens"]:
        print(f"     {tok:22s} [{tag}]")
    print("-"*60)

show("http://9779.info/%E5%84%BF%E7%AB%A5%E7%AB%8B%E4%BD%93%E7%BA%B8%E8%B4%B4%E7%94%BB/")
show("http://www.marketingbyinternet.com/mo/e56508df639f6ce7d55c81ee3fcd5ba8/")
show("http://www.lebensmittel-ueberwachung.de/index.php/aktuelles.1")

URL : http://9779.info/%E5%84%BF%E7%AB%A5%E7%AB%8B%E4%BD%93%E7%BA%B8%E8%B4%B4%E7%94%BB/
解碼: http://9779.info/儿童立体纸贴画/
翻譯: [('儿童立体纸贴画', "Children's stand-alone paper posters")]
後綴(TLD): 'info' | 網域: '9779' | 副檔名: None
帶標籤 token:
     http                   [scheme]
     9779                   [domain]
     info                   [tld]
     儿童                     [cjk]
     立体                     [cjk]
     纸                      [cjk]
     贴画                     [cjk]
     children               [label_en]
     s                      [label_en]
     stand                  [label_en]
     alone                  [label_en]
     paper                  [label_en]
     posters                [label_en]
------------------------------------------------------------


URL : http://www.marketingbyinternet.com/mo/e56508df639f6ce7d55c81ee3fcd5ba8/
解碼: http://www.marketingbyinternet.com/mo/e56508df639f6ce7d55c81ee3fcd5ba8/
後綴(TLD): 'com' | 網域: 'marketingbyinternet' | 副檔名: None
帶標籤 token:
     http                   [scheme]
     marketing              [domain]
     by                     [domain]
     internet               [domain]
     com                    [tld]
     mo                     [word]
     e56508df639f6ce7d55c81ee3fcd5ba8 [anomaly]
------------------------------------------------------------
URL : http://www.lebensmittel-ueberwachung.de/index.php/aktuelles.1
解碼: http://www.lebensmittel-ueberwachung.de/index.php/aktuelles.1
翻譯: [('lebensmittel-ueberwachung aktuelles', 'current food monitoring')]
後綴(TLD): 'de' | 網域: 'lebensmittel-ueberwachung' | 副檔名: '1'
帶標籤 token:
     http                   [scheme]
     lebensmittel-ueberwachung [domain]
     de                     [tld]
     1                      [file_ext]
     index                 

## ⑦ 批次特徵工程 featurize_dataframe

對一批 URL 跑完整 pipeline，產生 14 個特徵欄位。
其中 **`token_str`** 是空白分隔的 token 字串，可直接餵 `TfidfVectorizer`。
（這裡先不翻譯，跑得快；翻譯版在下一節對中文子集示範。）

In [9]:
import time
sample = df.sample(5000, random_state=42).reset_index(drop=True)
t0 = time.time()
feat = uf.featurize_dataframe(sample)          # translate=False，快
print(f"5000 筆耗時 {time.time()-t0:.1f}s（全量 651k 推估 ~{(time.time()-t0)/5000*651191/60:.0f} 分鐘）")
print("新增欄位:", [c for c in feat.columns if c not in df.columns])
feat[["type","suffix","file_ext","decode_encoding","has_cjk","n_anomaly","n_tokens","token_str"]].head(6)

5000 筆耗時 2.0s（全量 651k 推估 ~4 分鐘）
新增欄位: ['domain', 'suffix', 'file_ext', 'decode_encoding', 'was_encoded', 'token_str', 'n_tokens', 'n_anomaly', 'url_len', 'url_entropy', 'has_cjk', 'has_ip_host', 'n_digits', 'n_special', 'malformed']


,type,suffix,file_ext,decode_encoding,has_cjk,n_anomaly,n_tokens,token_str
0,malware,,sh4,NaN,False,0,6,37.49.226.178 sh4 deus bins deus sh4
1,benign,com,NaN,NaN,False,1,5,thefreedictionary com Galt tre phine
2,phishing,com,NaN,NaN,False,0,4,jscape com ssh factory
3,defacement,org.au,NaN,NaN,False,0,8,wsnc org.au component j cal pro view 983
4,benign,com,html,NaN,False,0,18,virtualtourist com html travel North america C...
5,benign,com,NaN,NaN,False,0,6,evri com person donald ballard 0x87e96


In [10]:
# 這些 token 特徵對「分類」有沒有鑑別力？看各類別平均
display(feat.groupby("type")[["n_anomaly","url_entropy","n_tokens","has_cjk","url_len"]].mean().round(2))
print("\n含中文(has_cjk)的編碼 URL 幾乎只出現在 malware → 解碼本身就是有用特徵")
print("\n後綴(TLD) Top10:")
print(feat["suffix"].value_counts().head(10))

,n_anomaly,url_entropy,n_tokens,has_cjk,url_len
type,,,,,
benign,0.19,4.23,9.65,0.0,60.53
defacement,0.07,4.46,14.18,0.0,86.64
malware,0.17,4.24,8.03,0.1,59.00
phishing,0.19,4.03,6.88,0.0,47.46



含中文(has_cjk)的編碼 URL 幾乎只出現在 malware → 解碼本身就是有用特徵

後綴(TLD) Top10:
suffix
com       3044
org        403
net        239
           105
co.uk       87
de          81
ca          77
edu         74
it          55
com.br      55
Name: count, dtype: int64


## ⑧ 翻譯版示範（只對含中文的子集，才不會慢）

In [11]:
cjk_rows = feat[feat["has_cjk"]].head(15).copy()
cjk_feat = uf.featurize_dataframe(cjk_rows[["url","type"]].reset_index(drop=True), translate=True)
cjk_feat[["type","translations","token_str"]]

,type,translations,token_str
0,malware,关于工作的剪贴画->Summary of work,9779 info 关于 工作 的 剪贴画 summary of work
1,malware,豆子粘贴画->A shoe box painting.,9779 info 豆子 粘贴 画 a shoe box painting
2,benign,業務最前線->Jobs & Frontline; 發掘客戶施力點->Sell custome...,udn com news story 7244 901088 業務 最 前線 發掘 客戶 施...
3,malware,剪贴画图片大->Crop image size,9779 info 剪贴画 图片 大 crop image size
4,malware,碎卡纸粘贴画->A piece of paper.,9779 info 碎 卡纸 粘贴 画 a piece of paper
5,malware,美丽的夜晚剪贴画->Beautiful night painting.,9779 info 美丽 的 夜晚 剪贴画 beautiful night painting
6,malware,昆曲剪贴画->Skew drawing,9779 info 昆曲 剪贴画 skew drawing
7,malware,大班种子粘贴画图片->Photo of the great breeder.,9779 info 大 班 种子 粘贴 画 图片 photo of the great br...
8,malware,彩纸立体粘贴画->Colorful paper cover art,9779 info 彩纸 立体 粘贴 画 colorful paper cover art
9,malware,关于天气的剪贴画->Cuts on the air,9779 info 关于 天气 的 剪贴画 cuts on the air


## ⑨ 下一步：把 token 特徵餵進分類器（baseline）

最低成本的做法：`token_str` → TF-IDF → 邏輯回歸。先看 4 分類 baseline 表現。

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 用前面 5000 筆的 token_str 當特徵（實務請用全量）
X_tr, X_te, y_tr, y_te = train_test_split(
    feat["token_str"], feat["type"], test_size=0.25, random_state=0, stratify=feat["type"])
vec = TfidfVectorizer(min_df=2, ngram_range=(1,1))
Xtr = vec.fit_transform(X_tr); Xte = vec.transform(X_te)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, y_tr)
print("特徵維度(詞彙量):", Xtr.shape[1])
print(classification_report(y_te, clf.predict(Xte)))

特徵維度(詞彙量): 3126
              precision    recall  f1-score   support

      benign       0.89      0.82      0.85       820
  defacement       0.75      0.83      0.79       186
     malware       0.88      0.80      0.84        64
    phishing       0.46      0.60      0.52       180

    accuracy                           0.79      1250
   macro avg       0.74      0.76      0.75      1250
weighted avg       0.81      0.79      0.79      1250



## ⑨b 全量資料完整評估（TF-IDF 詞+字元 + 數值特徵 → LinearSVC）

上面是 5000 筆的小 baseline。這裡直接讀**全量特徵檔** `data/url_features.csv`(651k)，
用更強的特徵：**詞 1-2gram + 字元 3-5gram**（字元 n-gram 對 URL 特別有效）＋ 9 個數值特徵，
看真正的 accuracy / per-class F1 / 混淆矩陣。

> ⚠️ 注意：`defacement` 通常會接近完美(F1~0.99)，這多半反映**資料來源特徵洩漏**（該子集來源單一、結構雷同），
> 實際部署不會這麼漂亮。**真正能代表模型實力的是 phishing 的 F1**（最難、也最重要）。

In [13]:
import time
import numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

d = pd.read_csv("./data/url_features.csv")
d["token_str"] = d["token_str"].fillna("")
y = d["type"]
tr, te = train_test_split(d.index, test_size=0.2, random_state=42, stratify=y)

# 文字特徵：詞 + 字元 n-gram（都建在排除 scheme 的 token_str 上）
w_vec = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3, sublinear_tf=True, max_features=150000)
c_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=5, sublinear_tf=True, max_features=200000)
Xw, Xwt = w_vec.fit_transform(d.loc[tr,"token_str"]), w_vec.transform(d.loc[te,"token_str"])
Xc, Xct = c_vec.fit_transform(d.loc[tr,"token_str"]), c_vec.transform(d.loc[te,"token_str"])

# 數值特徵
num_cols = ["n_anomaly","url_entropy","has_cjk","has_ip_host","n_digits","n_special","n_tokens","was_encoded","url_len"]
num = d[num_cols].fillna(0).astype(float)
sc = MaxAbsScaler()
Xtr = hstack([Xw, Xc, csr_matrix(sc.fit_transform(num.loc[tr]))]).tocsr()
Xte = hstack([Xwt, Xct, csr_matrix(sc.transform(num.loc[te]))]).tocsr()
print("特徵維度:", Xtr.shape[1])

t0 = time.time()
clf = LinearSVC(class_weight="balanced", C=1.0).fit(Xtr, y.loc[tr])
pred = clf.predict(Xte)
yte = y.loc[te]
print(f"訓練+預測 {time.time()-t0:.0f}s\n")
print(f"Accuracy    : {accuracy_score(yte,pred):.4f}")
print(f"Macro F1    : {f1_score(yte,pred,average='macro'):.4f}")
print(f"Weighted F1 : {f1_score(yte,pred,average='weighted'):.4f}\n")
print(classification_report(yte, pred, digits=3))

labels = sorted(y.unique())
cm = confusion_matrix(yte, pred, labels=labels)
print("混淆矩陣 (列=真實, 行=預測):")
display(pd.DataFrame(cm, index=labels, columns=labels))

特徵維度: 350009


訓練+預測 147s

Accuracy    : 0.9439


Macro F1    : 0.9368


Weighted F1 : 0.9445



              precision    recall  f1-score   support

      benign      0.964     0.954     0.959     85621
  defacement      0.995     0.998     0.997     19292
     malware      0.986     0.966     0.976      6504
    phishing      0.796     0.837     0.816     18822

    accuracy                          0.944    130239
   macro avg      0.935     0.939     0.937    130239
weighted avg      0.945     0.944     0.944    130239

混淆矩陣 (列=真實, 行=預測):


,benign,defacement,malware,phishing
benign,81644,61,37,3879
defacement,14,19259,0,19
malware,82,1,6281,140
phishing,2982,37,49,15754


## ⑨c 改進模型：加入 12 個釣魚線索特徵

用 `uf.extra_features()` 把釣魚網址的典型線索（子網域數、誘餌字 login/secure/paypal、可疑 TLD、`@`、IP 當主機…）
加進 ⑨b 的模型，看 phishing 能不能再往上。

In [14]:
import numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, accuracy_score

# 12 個釣魚線索特徵（從 url 直接算，很快）
EX = np.array([list(uf.extra_features(u).values()) for u in d["url"].astype(str)], dtype=float)
old_cols = ["n_anomaly","url_entropy","has_cjk","has_ip_host","n_digits","n_special","n_tokens","was_encoded","url_len"]
NUM = np.hstack([d[old_cols].fillna(0).astype(float).values, EX])      # 9 舊 + 12 新 = 21 數值

gi = d.index.get_indexer
w2 = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3, sublinear_tf=True, max_features=150000).fit(d.loc[tr,"token_str"])
c2 = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=5, sublinear_tf=True, max_features=200000).fit(d.loc[tr,"token_str"])
sc2 = MaxAbsScaler().fit(NUM[gi(tr)])
def build(idx):
    return hstack([w2.transform(d.loc[idx,"token_str"]), c2.transform(d.loc[idx,"token_str"]),
                   csr_matrix(sc2.transform(NUM[gi(idx)]))]).tocsr()

clf2 = LinearSVC(class_weight="balanced", C=1.0).fit(build(tr), y.loc[tr])
pred2 = clf2.predict(build(te))
print(f"⑨b(無釣魚特徵)  phishing_F1 ≈ 0.816")
print(f"⑨c(加釣魚特徵)  Acc={accuracy_score(y.loc[te],pred2):.4f}  MacroF1={f1_score(y.loc[te],pred2,average='macro'):.4f}  phishing_F1={f1_score(y.loc[te],pred2,labels=['phishing'],average=None)[0]:.4f}\n")
print(classification_report(y.loc[te], pred2, digits=3))

⑨b(無釣魚特徵)  phishing_F1 ≈ 0.816


⑨c(加釣魚特徵)  Acc=0.9501  MacroF1=0.9432  phishing_F1=0.8354



              precision    recall  f1-score   support

      benign      0.968     0.958     0.963     85621
  defacement      0.995     0.999     0.997     19292
     malware      0.988     0.967     0.977      6504
    phishing      0.815     0.857     0.835     18822

    accuracy                          0.950    130239
   macro avg      0.942     0.945     0.943    130239
weighted avg      0.951     0.950     0.951    130239



## ⑩ 對照預訓練深度模型 URLBERT

拿現成的 `CrabInHoney/urlbert-tiny-v4-malicious-url-classifier`（在數百萬條 URL 上預訓練、預測同樣 4 類）
跟我們的模型在**同一批 8000 筆**測試樣本上公平對照。

> ⚠️ **重要**：URLBERT 幾乎確定是用你這份 Kaggle 資料訓練的（**包含你的測試列**）→ 它的分數會「考前看過考卷」般虛高；
> 我們的模型只用 80% 訓練、在沒看過的資料上評估，是**誠實分數**。標籤對應取自模型卡 README。

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 測試集分層抽 8000 筆（每類 2000）
parts = [g.sample(min(2000, len(g)), random_state=0) for _, g in d.loc[te].groupby("type")]
samp = pd.concat(parts); ytrue = samp["type"].tolist()

pred_ours = clf2.predict(build(samp.index))           # 重用 ⑨c 訓好的模型

ID2 = {0:"benign", 1:"defacement", 2:"malware", 3:"phishing"}    # 來自模型卡 README label_mapping
M = "CrabInHoney/urlbert-tiny-v4-malicious-url-classifier"
tok = AutoTokenizer.from_pretrained(M)
bert = AutoModelForSequenceClassification.from_pretrained(M).eval()
urls = samp["url"].astype(str).tolist(); pred_bert = []
with torch.no_grad():
    for i in range(0, len(urls), 256):
        b = tok(urls[i:i+256], return_tensors="pt", padding=True, truncation=True, max_length=64)   # 此模型最大長度=64
        pred_bert += [ID2[p] for p in bert(**b).logits.argmax(-1).tolist()]

rows = []
for name, pred in [("我們的 TF-IDF+特徵 (誠實 held-out)", pred_ours), ("URLBERT (疑似看過答案)", pred_bert)]:
    rows.append({"模型": name,
                 "Accuracy": round(accuracy_score(ytrue, pred), 4),
                 "MacroF1": round(f1_score(ytrue, pred, average="macro"), 4),
                 "phishing_F1": round(f1_score(ytrue, pred, labels=["phishing"], average=None)[0], 4)})
print("同一批 8000 筆測試樣本上的對照：")
display(pd.DataFrame(rows))

同一批 8000 筆測試樣本上的對照：


,模型,Accuracy,MacroF1,phishing_F1
0,我們的 TF-IDF+特徵 (誠實 held-out),0.9429,0.9430,0.8888
1,URLBERT (疑似看過答案),0.9741,0.9741,0.9510


## ⑪ 注意事項 (gotchas)

- **`http(s)://` scheme 是資料來源洩漏**：不同類別帶 scheme 的比例差很多，別讓它主導模型（我們已把 scheme 排除在 token 外）。
- **英文斷字庫不支援中文**：URL 裡的中文是 `%XX` 編碼 → 一定先 `smart_decode` 再用 jieba。
- **後綴別只取最後一段**：`bbc.co.uk` 後綴是 `co.uk`，務必用 `tldextract`。
- **熵門檻要校準**：預設母音護欄 + 熵 3.5，請依你的資料微調；短 token(<8字)熵不可靠。
- **翻譯的取捨**：非拉丁語系(中日韓)一定翻；拉丁語系(德/西…)靠 langid 偵測，信心<0.90 不翻（如匈牙利文短字段可能漏翻，門檻可調）。翻譯較慢，建議只對 `has_cjk` 或偵測為非英文的列開 `translate=True`，並善用快取。

### 如何跑全量並存檔
```python
full = uf.featurize_dataframe(df, translate=False)        # ~4 分鐘
full.to_csv("./data/url_features.csv", index=False)
# 想含翻譯：只對需要的子集開 translate=True，再 merge 回去
```

In [16]:
full = uf.featurize_dataframe(df, translate=False)        # ~4 分鐘
full.to_csv("./data/url_features.csv", index=False)

In [17]:
full

,url,type,domain,suffix,file_ext,decode_encoding,was_encoded,token_str,n_tokens,n_anomaly,url_len,url_entropy,has_cjk,has_ip_host,n_digits,n_special,malformed
0,br-icloud.com.br,phishing,br-icloud,com.br,NaN,NaN,False,br-icloud com.br,2,0,16,3.375,False,False,0,3,False
1,mp3raid.com/music/krizz_kaliko.html,benign,mp3raid,com,html,NaN,False,mp3raid com html music krizz kaliko html,7,0,35,4.079,False,False,1,5,False
2,bopsecrets.org/rexroth/cr/1.htm,benign,bopsecrets,org,htm,NaN,False,bop secrets org htm rexroth cr 1 htm,8,0,31,3.708,False,False,1,5,False
3,http://www.garage-pirenne.be/index.php?option=...,defacement,garage-pirenne,be,php,NaN,False,garage-pirenne be php index php option com con...,15,0,88,4.660,False,False,7,18,False
4,http://adventure-nicaragua.net/index.php?optio...,defacement,adventure-nicaragua,net,php,NaN,False,adventure-nicaragua net php index php option c...,11,1,235,5.491,False,False,22,14,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
651186,xbox360.ign.com/objects/850/850402.html,phishing,ign,com,html,NaN,False,xbox360 ign com html objects 850 850402 html,8,0,39,4.356,False,False,12,6,False
651187,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing,teamxbox,com,NaN,NaN,False,games teamxbox com xbox 360 1860 Dead Space,8,0,44,4.243,False,False,7,8,False
651188,www.gamespot.com/xbox360/action/deadspace/,phishing,gamespot,com,NaN,NaN,False,gamespot com xbox360 action dead space,6,0,42,4.148,False,False,3,6,False
651189,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing,wikipedia,org,NaN,NaN,False,en wikipedia org wiki Dead Space video game,8,0,45,4.102,False,False,0,9,False
